# Task 12.1 – European Options: Black-Scholes-Merton with Greeks

In [1]:
import numpy as np
from scipy.stats import norm
import pandas as pd

In [2]:
def bsm_price_and_greeks(S, K, T, r, q, sigma, option_type):
    """
    Compute Black-Scholes-Merton price and Greeks for a European option.

    Parameters
    ----------
    S : float  - Current stock price
    K : float  - Strike price
    T : float  - Time to maturity in years
    r : float  - Continuously compounded risk-free rate
    q : float  - Continuously compounded dividend yield
    sigma : float - Implied volatility
    option_type : str - 'Call' or 'Put'

    Returns
    -------
    dict with keys: Value, Delta, Gamma, Vega, Rho, Theta
        Theta is annualised (per year, sign convention: negative = value lost as time passes)
        Vega  is per unit change in sigma (not per 1%)
    """
    sqrt_T = np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T

    e_qT = np.exp(-q * T)
    e_rT = np.exp(-r * T)
    n_d1 = norm.pdf(d1)

    # Gamma and Vega are the same for calls and puts
    gamma = e_qT * n_d1 / (S * sigma * sqrt_T)
    vega  = S * e_qT * n_d1 * sqrt_T

    if option_type.strip().lower() == 'call':
        N_d1  = norm.cdf(d1)
        N_d2  = norm.cdf(d2)
        value = S * e_qT * N_d1 - K * e_rT * N_d2
        delta = e_qT * N_d1
        rho   = K * T * e_rT * N_d2
        theta = (-S * e_qT * n_d1 * sigma / (2 * sqrt_T)
                 - r * K * e_rT * N_d2
                 + q * S * e_qT * N_d1)
    else:  # put
        N_nd1 = norm.cdf(-d1)
        N_nd2 = norm.cdf(-d2)
        value = K * e_rT * N_nd2 - S * e_qT * N_nd1
        delta = -e_qT * N_nd1
        rho   = -K * T * e_rT * N_nd2
        theta = (-S * e_qT * n_d1 * sigma / (2 * sqrt_T)
                 + r * K * e_rT * N_nd2
                 - q * S * e_qT * N_nd1)

    return {"Value": value, "Delta": delta, "Gamma": gamma,
            "Vega": vega, "Rho": rho, "Theta": theta}

In [3]:
def process_bsm(input_path, output_path):
    """
    Read option parameters from input_path, compute BSM price and Greeks,
    print results, and save to output_path.
    """
    df = pd.read_csv(input_path).dropna(subset=["ID"])
    df["ID"] = df["ID"].astype(int)

    results = []
    for _, row in df.iterrows():
        T = row["DaysToMaturity"] / row["DayPerYear"]
        greeks = bsm_price_and_greeks(
            S=row["Underlying"],
            K=row["Strike"],
            T=T,
            r=row["RiskFreeRate"],
            q=row["DividendRate"],
            sigma=row["ImpliedVol"],
            option_type=row["Option Type"],
        )
        results.append({"ID": int(row["ID"]), **greeks})

    out = pd.DataFrame(results, columns=["ID", "Value", "Delta", "Gamma", "Vega", "Rho", "Theta"])
    print(out.to_string(index=False))
    out.to_csv(output_path, index=False)
    print(f"\nSaved to {output_path}")
    return out

results_12_1 = process_bsm("test12_1.csv", "testout_12.1.csv")

 ID     Value     Delta    Gamma      Vega        Rho      Theta
  1  3.260824  0.547872 0.053506 14.659079   7.058414 -13.019817
  2  2.646281 -0.452128 0.053506 14.659079  -6.556032  -8.547471
  3 22.043329  0.685508 0.008115 35.571438  50.967099  -7.213608
  4 20.449083 -0.470751 0.009310 40.812329 -73.999110  -5.351164

Saved to testout_12.1.csv
